In [ ]:
# Concatenate a behavior session and an optotagging session recorded separately
# with Open Ephys into a single continuous recording, so both can be spike-sorted
# together (units are shared across the two epochs).
#
# Output: one preprocessed (phase-shift + CAR) binary ready for Kilosort, the
# concatenated ADC stream (for event detection across both epochs), and a
# session_boundaries.json recording where each session starts/ends in the
# concatenated sample/time axis — needed later to split spike times and events
# back out by session.

In [ ]:
# ── paths and params ───────────────────────────────────────────────────────────
from pathlib import Path

# order matters: recordings are concatenated in this order
session_paths = {
    "behavior":    Path(r"E:\D1-1-4_IM-1971\ephys_raw\2026-06-19_14-59-14"),
    "optotagging": Path(r"E:\D1-1-4_IM-1971\ephys_raw\2026-06-19_15-40-02"),
}

PROBE_IDX   = 0       # which ephys stream to concatenate
STREAM_NAME = None    # e.g. "ProbeA"; None = auto-detect from first session's OE folder

# output: concatenated + preprocessed data will be saved here
output_dir = session_paths["behavior"].parent / "concatenated"

# preprocessing options (applied after concatenation, same as 0_preprocess_phase_shift.py)
DO_CAR   = True       # common average reference after phase shift (recommended)
CAR_MODE = "global"   # "global" or "local"
CAR_OP   = "median"   # "median" or "average"

In [ ]:
# ── parse OE folders and check the two sessions are compatible ────────────────
import sys
sys.path.append(str(Path(__file__).parent.parent) if "__file__" in dir() else "..")

from src import oe_parse_folders, oe_parse_params

session_names = list(session_paths)  # preserves insertion order

oe_names_by_session  = {}
oe_paths_by_session  = {}
probe_params_by_session = {}
adc_params_by_session    = {}

for name in session_names:
    oe_names, oe_paths = oe_parse_folders(session_paths[name])
    probes_params, adc_params = oe_parse_params(oe_paths["xml"], oe_paths["oebin"])

    oe_names_by_session[name] = oe_names
    oe_paths_by_session[name] = oe_paths
    probe_params_by_session[name] = probes_params[PROBE_IDX]
    adc_params_by_session[name]   = adc_params

stream_name = STREAM_NAME or probe_params_by_session[session_names[0]]["stream_name"]

# sanity check: same probe stream / channel count / sample rate across sessions
ref_params = probe_params_by_session[session_names[0]]
for name in session_names[1:]:
    p = probe_params_by_session[name]
    for key in ("stream_name", "channel_count", "sample_rate"):
        if str(p[key]) != str(ref_params[key]):
            raise ValueError(
                f"Session '{name}' probe param '{key}'={p[key]!r} does not match "
                f"'{session_names[0]}' ({ref_params[key]!r}). Sessions must use the "
                f"same probe configuration to be concatenated."
            )

print("Sessions to concatenate (in order):")
for name in session_names:
    print(f"  {name:12s}: {session_paths[name]}")
print(f"Stream : {stream_name}  ({ref_params['channel_count']} ch,  {ref_params['sample_rate']} Hz)")
print(f"Output : {output_dir}")

In [ ]:
# ── load each session and concatenate with SpikeInterface ─────────────────────
import spikeinterface.full as si

recordings = [
    si.read_openephys(session_paths[name], stream_name=stream_name)
    for name in session_names
]

# number of samples contributed by each session, in concatenation order
n_samples_by_session = {name: rec.get_num_samples() for name, rec in zip(session_names, recordings)}

recording_concat = si.concatenate_recordings(recordings)

print("Loaded and concatenated:")
for name, rec in zip(session_names, recordings):
    print(f"  {name:12s}: {rec.get_num_samples()} samples "
          f"({rec.get_num_samples() / rec.get_sampling_frequency():.1f} s)")
print(f"  {'total':12s}: {recording_concat.get_num_samples()} samples "
      f"({recording_concat.get_num_samples() / recording_concat.get_sampling_frequency():.1f} s)")

In [ ]:
# ── apply phase shift + CAR on the concatenated recording ─────────────────────
recording_ps = si.phase_shift(recording_concat)
print("Phase shift correction applied.")

if DO_CAR:
    recording_pre = si.common_reference(recording_ps, reference=CAR_MODE, operator=CAR_OP)
    print(f"CAR applied: {CAR_MODE} {CAR_OP}.")
else:
    recording_pre = recording_ps

In [ ]:
# ── save concatenated + preprocessed ephys binary ──────────────────────────────
output_dir.mkdir(parents=True, exist_ok=True)

recording_pre.save_to_folder(folder=output_dir, overwrite=True,
                             chunk_duration="1s", n_jobs=4,
                             progress_bar=True)

dat_file   = output_dir / "traces_cached_seg0.raw"
n_chan_bin = recording_pre.get_num_channels()
sample_rate = float(recording_pre.get_sampling_frequency())

In [ ]:
# ── concatenate the ADC stream too (needed to detect events across both epochs) ─
import numpy as np
from src import oe_load_adc

adc_data_by_session = {}
adc_t_by_session     = {}
for name in session_names:
    data, t_arr, adc_fs = oe_load_adc(oe_paths_by_session[name]["adc_stream"], adc_params_by_session[name])
    adc_data_by_session[name] = data
    adc_t_by_session[name]    = t_arr

# stitch ADC traces end to end; rebuild a monotonic timestamp axis using the
# concatenated ephys sample rate as the common clock (OE timestamps within a
# session are already contiguous, only the jump between sessions needs fixing)
adc_data_concat = np.concatenate([adc_data_by_session[name] for name in session_names], axis=0)

t_arr_parts = []
t_offset = 0.0
for name in session_names:
    t_local = adc_t_by_session[name] - adc_t_by_session[name][0] + t_offset
    t_arr_parts.append(t_local)
    t_offset = t_local[-1] + 1.0 / adc_fs
adc_t_concat = np.concatenate(t_arr_parts)

np.save(output_dir / "adc_continuous.npy", adc_data_concat)
np.save(output_dir / "adc_timestamps.npy", adc_t_concat)

print(f"Concatenated ADC saved: {adc_data_concat.shape[0]} samples, {adc_data_concat.shape[1]} channels")

In [ ]:
# ── save session boundaries (sample index + time offset in the concatenated axis) ─
import json

boundaries = []
sample_cursor = 0
for name in session_names:
    n = n_samples_by_session[name]
    boundaries.append({
        "session": name,
        "path": str(session_paths[name]),
        "sample_start": sample_cursor,
        "sample_end": sample_cursor + n,          # exclusive
        "t_start_s": sample_cursor / sample_rate,
        "t_end_s": (sample_cursor + n) / sample_rate,
        "n_samples": n,
    })
    sample_cursor += n

meta = {
    "stream_name": stream_name,
    "sample_rate": sample_rate,
    "n_chan_bin": n_chan_bin,
    "session_order": session_names,
    "sessions": boundaries,
}
with open(output_dir / "session_boundaries.json", "w") as f:
    json.dump(meta, f, indent=2)

print(f"\nDone.")
print(f"Binary               : {dat_file}")
print(f"n_chan_bin           : {n_chan_bin}")
print(f"Session boundaries   : {output_dir / 'session_boundaries.json'}")
for b in boundaries:
    print(f"  {b['session']:12s}: samples [{b['sample_start']}, {b['sample_end']})  "
          f"t = [{b['t_start_s']:.1f}, {b['t_end_s']:.1f}] s")
print(f"\nIn 2.2_run_kilosort.py, set:")
print(f"  dat_file  = Path(r'{dat_file}')")
print(f"  n_chan_bin = {n_chan_bin}")
print(f"  do_CAR    = False          # CAR already applied here")
print(f"\nAfter sorting, use session_boundaries.json to split spike_times.npy")
print(f"(sample indices) by session.")